# Q(V, φ) parametrization

Collected charge **Q** of ETL LGADs vs bias **V** and fluence **φ**. Data is read directly from `parametrization_data.xlsx` (one sheet per production).

Modules: `config.py` (settings) · `data_io.py` (Excel reader) · `model.py` (fits) · `plots.py` (ROOT plots).

**Model.** Per fluence `Q(V) = p0 + p1·V + exp(p2 + p3·V)` (analytically seeded → no manual tuning). Each parameter vs φ: `param_mode='auto'` = linear if < 4 fluences, else exp/lin per parameter by AIC. Charge error is set in `config.ERR_Q` (default 1 fC, switchable to 2).

Kernel: **Python (tb_analysis2)**.

In [ ]:
import ROOT
%jsroot on
import importlib
import config, data_io, model, plots
for m in (config, data_io, model, plots):
    importlib.reload(m)

# charge error: default 1 fC, switchable to 2
config.set_charge_error(1.0)

sensors = data_io.load_sensors()
print(data_io.summary(sensors))
print('\ncharge error = +-%.1f fC | param_mode = %s' % (config.ERR_Q, config.PARAM_MODE))

In [ ]:
# fit every production (sheet). param_mode: 'auto' | 'lin' | 'exp' | 'interp'
results = model.fit_all(sensors, verbose=True)

## Q(V) fits — pooled data + fit (open red = test beam, cross-check only)

In [ ]:
for name, r in results.items():
    plots.plot_QV(sensors, r).Draw()

## Parameters vs fluence (blue = law used; dashed grey = linear trend)

In [ ]:
for name, r in results.items():
    plots.plot_params(r).Draw()

## Prediction check — smooth Q(V,φ) surface vs data

In [ ]:
for name, r in results.items():
    plots.plot_prediction(r).Draw()

## Bias to reach the target charge, vs fluence

In [ ]:
plots.plot_bias_for_target(results, Q_target=config.Q_TARGET).Draw()
print('Bias to reach Q = %.0f fC:' % config.Q_TARGET)
for name, r in results.items():
    row = {}
    for phi in r['fluences']:
        v = model.V_for_Q(config.Q_TARGET, phi, r['coeffs'])
        row['%.1e' % phi] = 'n/a' if v != v else '%.0f V' % v
    print(' ', name, row)

## Closed-form Q(V,φ) surface (TF2)

In [ ]:
surfaces = {name: model.build_QVphi_TF2(r['coeffs'], 'TF2_' + model._short(name))
            for name, r in results.items()}
c = ROOT.TCanvas('csurf', 'surfaces', 500 * len(surfaces), 450)
c.Divide(len(surfaces), 1)
for i, (name, f2) in enumerate(surfaces.items()):
    c.cd(i + 1); ROOT.gPad.SetRightMargin(0.16); f2.SetTitle(name); f2.Draw('colz')
c.Draw()